# Knowledge Graph for Dietary Recommendations in Disease Management
**Course:** Knowledge Graphs — TU Wien 2026S  
**Author:** Serkan Besim

This notebook implements a Knowledge Graph that recommends foods based on diseases.  
It combines two data sources (USDA FoodData Central and CTD), applies logical reasoning  
to infer dietary recommendations, and uses RotatE embeddings to predict missing food-disease links.

## 1. Setup
Import required libraries and verify installation.

In [1]:
import pandas as pd
import rdflib
import owlrl
from pykeen.pipeline import pipeline

In [2]:
# Key nutrients we care about
key_nutrients = {
    1003: 'Protein',
    1004: 'Fat',
    1005: 'Carbohydrate',
    1079: 'Fiber',
    1087: 'Calcium',
    1089: 'Iron',
    1090: 'Magnesium',
    1092: 'Potassium',
    1093: 'Sodium',
    1095: 'Zinc',
    1162: 'Vitamin_C',
    1114: 'Vitamin_D',
    1175: 'Vitamin_B6',
    1178: 'Vitamin_B12',
    1177: 'Folate',
    1253: 'Cholesterol',
    1063: 'Sugar',
    1258: 'Saturated_Fat',
}

## 2. Data Loading
We use two datasets:
- **USDA FoodData Central** — 365 foundation foods with nutrient compositions
- **Comparative Toxicogenomics Database (CTD)** — curated therapeutic nutrient-disease associations

In [3]:
# Load the three USDA files
food_df = pd.read_csv('C:/kg_project/food.csv')
nutrient_df = pd.read_csv('C:/kg_project/nutrient.csv')
food_nutrient_df = pd.read_csv('C:/kg_project/food_nutrient.csv', low_memory=False)
foundation_df = pd.read_csv('C:/kg_project/foundation_food.csv')

print("Foods:", food_df.shape)
print("Nutrients:", nutrient_df.shape)
print("Food-Nutrient:", food_nutrient_df.shape)
print("Foundation:", foundation_df.shape)

Foods: (78026, 5)
Nutrients: (477, 5)
Food-Nutrient: (159285, 11)
Foundation: (365, 3)


## 3. Data Preparation
Select 18 clinically relevant nutrients, filter to foundation foods,  
and label each food-nutrient pair as high/medium/low using quartile thresholds.

In [4]:
# Filter to only key nutrients
nutrient_filtered = nutrient_df[nutrient_df['id'].isin(key_nutrients.keys())].copy()
nutrient_filtered['clean_name'] = nutrient_filtered['id'].map(key_nutrients)
print(nutrient_filtered[['id', 'name', 'clean_name']])
print("\nTotal key nutrients:", len(nutrient_filtered))

       id                            name     clean_name
4    1003                         Protein        Protein
5    1004               Total lipid (fat)            Fat
6    1005     Carbohydrate, by difference   Carbohydrate
64   1063                   Sugars, Total          Sugar
80   1079            Fiber, total dietary          Fiber
88   1087                     Calcium, Ca        Calcium
90   1089                        Iron, Fe           Iron
91   1090                   Magnesium, Mg      Magnesium
93   1092                    Potassium, K      Potassium
94   1093                      Sodium, Na         Sodium
96   1095                        Zinc, Zn           Zinc
115  1114             Vitamin D (D2 + D3)      Vitamin_D
163  1162  Vitamin C, total ascorbic acid      Vitamin_C
176  1175                     Vitamin B-6     Vitamin_B6
178  1177                   Folate, total         Folate
179  1178                    Vitamin B-12    Vitamin_B12
254  1253                     C

In [5]:
# Get food names for foundation foods only
foundation_foods = food_df[food_df['fdc_id'].isin(foundation_df['fdc_id'])][['fdc_id', 'description']]
print(f"Foundation foods: {len(foundation_foods)}")

Foundation foods: 365


In [6]:
# Join foundation foods with their nutrient values
food_nutrient_filtered = food_nutrient_df[
    (food_nutrient_df['fdc_id'].isin(foundation_foods['fdc_id'])) &
    (food_nutrient_df['nutrient_id'].isin(key_nutrients.keys()))
][['fdc_id', 'nutrient_id', 'amount']]

# Add food names
food_nutrient_filtered = food_nutrient_filtered.merge(
    foundation_foods, on='fdc_id'
)

# Add nutrient names
food_nutrient_filtered['nutrient_name'] = food_nutrient_filtered['nutrient_id'].map(key_nutrients)

# Final clean dataframe
df = food_nutrient_filtered[['fdc_id', 'description', 'nutrient_name', 'amount']].copy()
df.columns = ['fdc_id', 'food', 'nutrient', 'amount']

print(df.shape)
print(df.head(20))

(4189, 4)
    fdc_id                  food       nutrient   amount
0   321358    Hummus, commercial      Magnesium   71.100
1   321358    Hummus, commercial  Saturated_Fat    2.220
2   321358    Hummus, commercial           Iron    2.410
3   321358    Hummus, commercial     Vitamin_B6    0.143
4   321358    Hummus, commercial          Fiber    5.400
5   321358    Hummus, commercial          Sugar    0.340
6   321358    Hummus, commercial        Protein    7.350
7   321358    Hummus, commercial        Calcium   41.000
8   321358    Hummus, commercial      Potassium  289.000
9   321358    Hummus, commercial         Sodium  438.000
10  321358    Hummus, commercial         Folate   36.000
11  321358    Hummus, commercial            Fat   17.100
12  321358    Hummus, commercial           Zinc    1.380
13  321358    Hummus, commercial   Carbohydrate   14.900
14  321358    Hummus, commercial      Vitamin_C    0.000
15  321360  Tomatoes, grape, raw      Vitamin_C   27.200
16  321360  Tomatoes,

In [7]:
# Calculate thresholds per nutrient
thresholds = df.groupby('nutrient')['amount'].quantile([0.25, 0.75]).unstack()
thresholds.columns = ['low_threshold', 'high_threshold']
print(thresholds.round(2))

               low_threshold  high_threshold
nutrient                                    
Calcium                 9.22           72.80
Carbohydrate            2.54           20.14
Cholesterol            53.67           92.00
Fat                     0.32            7.76
Fiber                   1.80            4.45
Folate                  8.99           56.31
Iron                    0.14            2.17
Magnesium              11.90           44.60
Potassium             160.70          375.90
Protein                 1.12           18.19
Saturated_Fat           1.06           11.24
Sodium                  0.59          111.45
Sugar                   2.26            8.88
Vitamin_B12             0.57            1.66
Vitamin_B6              0.06            0.20
Vitamin_C               4.60           34.97
Vitamin_D               0.00            1.15
Zinc                    0.21            2.72


In [8]:
# Add level labels to each food-nutrient pair
def get_level(row):
    high = thresholds.loc[row['nutrient'], 'high_threshold']
    low = thresholds.loc[row['nutrient'], 'low_threshold']
    if row['amount'] >= high:
        return 'high'
    elif row['amount'] <= low:
        return 'low'
    else:
        return 'medium'

df['level'] = df.apply(get_level, axis=1)

print(df.head(20))
print("\nLevel distribution:")
print(df['level'].value_counts())

    fdc_id                  food       nutrient   amount   level
0   321358    Hummus, commercial      Magnesium   71.100    high
1   321358    Hummus, commercial  Saturated_Fat    2.220  medium
2   321358    Hummus, commercial           Iron    2.410    high
3   321358    Hummus, commercial     Vitamin_B6    0.143  medium
4   321358    Hummus, commercial          Fiber    5.400    high
5   321358    Hummus, commercial          Sugar    0.340     low
6   321358    Hummus, commercial        Protein    7.350  medium
7   321358    Hummus, commercial        Calcium   41.000  medium
8   321358    Hummus, commercial      Potassium  289.000  medium
9   321358    Hummus, commercial         Sodium  438.000    high
10  321358    Hummus, commercial         Folate   36.000  medium
11  321358    Hummus, commercial            Fat   17.100    high
12  321358    Hummus, commercial           Zinc    1.380  medium
13  321358    Hummus, commercial   Carbohydrate   14.900  medium
14  321358    Hummus, com

## 4. Knowledge Graph Construction
Translate the prepared data into RDF triples using rdflib.  
Each food-nutrient pair becomes a triple, e.g.:  
`(Lentils_dry) -- highIn --> (Iron)`

In [9]:
from rdflib import Graph, Namespace, RDF, RDFS, Literal, URIRef
import re

# Define namespaces
FOOD = Namespace("http://nutrition-kg.org/food/")
NUTRIENT = Namespace("http://nutrition-kg.org/nutrient/")
PROP = Namespace("http://nutrition-kg.org/property/")

def clean_uri(name):
    name = name.replace(' ', '_')
    name = re.sub(r'[^a-zA-Z0-9_]', '', name)
    return name

# Build graph with clean URIs
g = Graph()
g.bind("food", FOOD)
g.bind("nutrient", NUTRIENT)
g.bind("prop", PROP)

for _, row in df.iterrows():
    food_uri = FOOD[clean_uri(row['food'])]
    nutrient_uri = NUTRIENT[row['nutrient']]
    relation = PROP[f"{row['level']}In"]
    g.add((food_uri, relation, nutrient_uri))

print(f"Triples in graph: {len(g)}")
print("\nSample triples:")
for s, p, o in list(g)[:5]:
    print(f"  {s.split('/')[-1]} -- {p.split('/')[-1]} --> {o.split('/')[-1]}")

Triples in graph: 4189

Sample triples:
  Grape_juice_white_with_added_vitamin_C_from_concentrate_shelf_stable -- lowIn --> Fat
  Cheese_cottage_lowfat_2_milkfat -- lowIn --> Vitamin_B6
  Tomato_puree_canned -- mediumIn --> Vitamin_C
  Beef_short_loin_tbone_steak_bonein_separable_lean_only_trimmed_to_18_fat_choice_cooked_grilled -- mediumIn --> Saturated_Fat
  Arugula_baby_raw -- highIn --> Folate


In [10]:
# Add type information
for _, row in df[['food']].drop_duplicates().iterrows():
    food_uri = FOOD[clean_uri(row['food'])]
    g.add((food_uri, RDF.type, PROP['Food']))

for nutrient_name in key_nutrients.values():
    nutrient_uri = NUTRIENT[nutrient_name]
    g.add((nutrient_uri, RDF.type, PROP['Nutrient']))

print(f"Total triples after adding types: {len(g)}")

Total triples after adding types: 4572


## 5. Enrichment with Disease Data
Load therapeutic associations from CTD and add nutrient-disease triples:  
`(Iron) -- treats --> (Anemia)`

In [11]:
# Load CTD data
ctd = pd.read_csv('C:/kg_project/CTD_curated_chemicals_diseases.csv', 
                  comment='#',  # skip header comment lines
                  names=['ChemicalName', 'ChemicalID', 'CasRN', 
                         'DiseaseName', 'DiseaseID', 'DirectEvidence', 'PubMedIDs'])

In [12]:
# Keep only therapeutic relationships
ctd_therapeutic = ctd[ctd['DirectEvidence'] == 'therapeutic'].copy()
print(f"Therapeutic associations: {len(ctd_therapeutic)}")

# See if any of our nutrients appear as chemicals
our_nutrients = list(key_nutrients.values())
print("\nOur nutrients:", our_nutrients)

# Check which of our nutrients appear in CTD
ctd_therapeutic['ChemicalName_lower'] = ctd_therapeutic['ChemicalName'].str.lower()
matches = ctd_therapeutic[ctd_therapeutic['ChemicalName_lower'].isin(
    [n.lower().replace('_', ' ') for n in our_nutrients]
)]

Therapeutic associations: 39568

Our nutrients: ['Protein', 'Fat', 'Carbohydrate', 'Fiber', 'Calcium', 'Iron', 'Magnesium', 'Potassium', 'Sodium', 'Zinc', 'Vitamin_C', 'Vitamin_D', 'Vitamin_B6', 'Vitamin_B12', 'Folate', 'Cholesterol', 'Sugar', 'Saturated_Fat']


In [13]:
# Updated nutrient name mapping between our names and CTD names
nutrient_name_map = {
    'Zinc': 'Zinc',
    'Vitamin_D': 'Vitamin D',
    'Calcium': 'Calcium',
    'Magnesium': 'Magnesium',
    'Potassium': 'Potassium',
    'Sodium': 'Sodium',
    'Iron': 'Iron',
    'Fiber': 'Dietary Fiber',
}

# Get all matches with correct names
all_matches = []
for our_name, ctd_name in nutrient_name_map.items():
    results = ctd_therapeutic[
        ctd_therapeutic['ChemicalName'] == ctd_name
    ].copy()
    results['our_nutrient_name'] = our_name
    all_matches.append(results)

matches_final = pd.concat(all_matches)
print(f"Total therapeutic associations: {len(matches_final)}")
print(matches_final.groupby('our_nutrient_name')['DiseaseName'].count().sort_values(ascending=False))

Total therapeutic associations: 200
our_nutrient_name
Zinc         60
Vitamin_D    40
Calcium      31
Magnesium    30
Potassium    21
Sodium       11
Iron          6
Fiber         1
Name: DiseaseName, dtype: int64


In [14]:
from rdflib import Namespace

DISEASE = Namespace("http://nutrition-kg.org/disease/")
g.bind("disease", DISEASE)

# Add disease triples to KG
for _, row in matches_final.iterrows():
    nutrient_uri = NUTRIENT[row['our_nutrient_name']]
    disease_name_clean = re.sub(r'[^a-zA-Z0-9_]', '_', row['DiseaseName'])
    disease_uri = DISEASE[disease_name_clean]
    
    # nutrient --treats--> disease
    g.add((nutrient_uri, PROP['treats'], disease_uri))
    
    # disease type triple
    g.add((disease_uri, RDF.type, PROP['Disease']))

print(f"Total triples now: {len(g)}")
print(f"\nSample disease triples:")
count = 0
for s, p, o in g.triples((None, PROP['treats'], None)):
    print(f"  {s.split('/')[-1]} -- treats --> {o.split('/')[-1]}")
    count += 1
    if count >= 10:
        break

Total triples now: 4925

Sample disease triples:
  Zinc -- treats --> Acrodermatitis
  Zinc -- treats --> Acute_Kidney_Injury
  Vitamin_D -- treats --> Acute_Kidney_Injury
  Calcium -- treats --> Acute_Kidney_Injury
  Magnesium -- treats --> Acute_Kidney_Injury
  Potassium -- treats --> Acute_Kidney_Injury
  Sodium -- treats --> Acute_Kidney_Injury
  Zinc -- treats --> Asthma
  Zinc -- treats --> Atherosclerosis
  Zinc -- treats --> Brain_Injuries


In [15]:
# Summary of the KG
foods = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Food']))))
nutrients = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Nutrient']))))
diseases = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Disease']))))
food_nutrient_triples = len(list(g.triples((None, PROP['highIn'], None)))) + \
                         len(list(g.triples((None, PROP['mediumIn'], None)))) + \
                         len(list(g.triples((None, PROP['lowIn'], None))))
treats_triples = len(list(g.triples((None, PROP['treats'], None))))

print("=== Knowledge Graph Summary ===")
print(f"Foods:                  {foods}")
print(f"Nutrients:              {nutrients}")
print(f"Diseases:               {diseases}")
print(f"Food-nutrient triples:  {food_nutrient_triples}")
print(f"Nutrient-disease triples: {treats_triples}")
print(f"Total triples:          {len(g)}")

=== Knowledge Graph Summary ===
Foods:                  365
Nutrients:              18
Diseases:               153
Food-nutrient triples:  4189
Nutrient-disease triples: 200
Total triples:          4925


In [16]:
# Save the KG to disk
g.serialize(destination='C:/kg_project/nutrition_kg.ttl', format='turtle')
print("KG saved to nutrition_kg.ttl")

KG saved to nutrition_kg.ttl


## 6. Logical Reasoning
Apply inference rules to derive new knowledge:

**Rule 1 — Recommendation:**  
IF food highIn Nutrient X AND Nutrient X treats Disease D → food recommendedFor D

**Rule 2 — Contraindication:** (This one is not really implemented) 
IF food highIn Nutrient X AND Nutrient X worsensWith Disease D → food contraindicatedFor D

**Rule 3 — Contradiction resolution:**  
IF food recommendedFor D AND food contraindicatedFor D → remove recommendedFor

In [17]:
import owlrl

# First let's add the rule manually using RDFS/OWL reasoning
# We do this by iterating through the graph and inferring new triples

recommended_count = 0

# For every food-nutrient-disease chain where food is highIn nutrient
for food_uri, _, nutrient_uri in g.triples((None, PROP['highIn'], None)):
    for _, _, disease_uri in g.triples((nutrient_uri, PROP['treats'], None)):
        # Infer: food recommendedFor disease
        g.add((food_uri, PROP['recommendedFor'], disease_uri))
        recommended_count += 1

print(f"New inferred triples: {recommended_count}")
print(f"Total triples now: {len(g)}")
print("\nSample inferred triples:")
count = 10
for s, p, o in g.triples((None, PROP['recommendedFor'], None)):
    print(f"  {s.split('/')[-1]} -- recommendedFor --> {o.split('/')[-1]}")
    count += 1
    if count >= 20:
        break

New inferred triples: 14684
Total triples now: 18154

Sample inferred triples:
  Hummus_commercial -- recommendedFor --> Acute_Kidney_Injury
  Nuts_almonds_dry_roasted_with_salt_added -- recommendedFor --> Acute_Kidney_Injury
  Egg_white_dried -- recommendedFor --> Acute_Kidney_Injury
  Seeds_sunflower_seed_kernels_dry_roasted_with_salt_added -- recommendedFor --> Acute_Kidney_Injury
  Mustard_prepared_yellow -- recommendedFor --> Acute_Kidney_Injury
  Egg_whole_dried -- recommendedFor --> Acute_Kidney_Injury
  Restaurant_Latino_pupusas_con_frijoles_pupusas_bean -- recommendedFor --> Acute_Kidney_Injury
  Bread_wholewheat_commercially_prepared -- recommendedFor --> Acute_Kidney_Injury
  Figs_dried_uncooked -- recommendedFor --> Acute_Kidney_Injury
  Beans_Dry_Medium_Red_0_moisture -- recommendedFor --> Acute_Kidney_Injury


In [18]:
# Add manual contraindication knowledge
contraindications = [
    ('Sodium', 'Hypertension'),
    ('Sodium', 'Heart_Failure'),
    ('Cholesterol', 'Atherosclerosis'),
    ('Saturated_Fat', 'Atherosclerosis'),
    ('Sugar', 'Diabetes_Mellitus'),
    ('Saturated_Fat', 'Obesity'),
    ('Sugar', 'Obesity'),
]

for nutrient_name, disease_name in contraindications:
    nutrient_uri = NUTRIENT[nutrient_name]
    disease_uri = DISEASE[disease_name]
    g.add((nutrient_uri, PROP['worsensWith'], disease_uri))
    g.add((disease_uri, RDF.type, PROP['Disease']))

print(f"Added {len(contraindications)} contraindication relationships")

# Now infer: if food highIn nutrient AND nutrient worsensWith disease
# THEN food contraindicatedFor disease
contra_count = 0
for food_uri, _, nutrient_uri in g.triples((None, PROP['highIn'], None)):
    for _, _, disease_uri in g.triples((nutrient_uri, PROP['worsensWith'], None)):
        g.add((food_uri, PROP['contraindicatedFor'], disease_uri))
        contra_count += 1

print(f"New contraindication triples inferred: {contra_count}")
print(f"Total triples now: {len(g)}")

# Show some examples
print("\nSample contraindication triples:")
count = 0
for s, p, o in g.triples((None, PROP['contraindicatedFor'], None)):
    print(f"  {s.split('/')[-1]} -- contraindicatedFor --> {o.split('/')[-1]}")
    count += 1
    if count >= 10:
        break

Added 7 contraindication relationships
New contraindication triples inferred: 314
Total triples now: 18469

Sample contraindication triples:
  Hummus_commercial -- contraindicatedFor --> Hypertension
  Beans_snap_green_canned_regular_pack_drained_solids -- contraindicatedFor --> Hypertension
  Frankfurter_beef_unheated -- contraindicatedFor --> Hypertension
  Nuts_almonds_dry_roasted_with_salt_added -- contraindicatedFor --> Hypertension
  Egg_whole_raw_frozen_pasteurized -- contraindicatedFor --> Hypertension
  Egg_white_raw_frozen_pasteurized -- contraindicatedFor --> Hypertension
  Egg_white_dried -- contraindicatedFor --> Hypertension
  Onion_rings_breaded_par_fried_frozen_prepared_heated_in_oven -- contraindicatedFor --> Hypertension
  Pickles_cucumber_dill_or_kosher_dill -- contraindicatedFor --> Hypertension
  Cheese_parmesan_grated -- contraindicatedFor --> Hypertension


In [19]:
# Find and remove ALL remaining contradictions properly
resolved = 0
contradictions_found = []

for food_uri, _, disease_uri in list(g.triples((None, PROP['recommendedFor'], None))):
    if (food_uri, PROP['contraindicatedFor'], disease_uri) in g:
        contradictions_found.append((food_uri, disease_uri))

print(f"Contradictions found: {len(contradictions_found)}")

# Remove recommendedFor where contraindication exists
for food_uri, disease_uri in contradictions_found:
    g.remove((food_uri, PROP['recommendedFor'], disease_uri))
    resolved += 1

print(f"Resolved: {resolved}")

# Verify
remaining = [
    (s, o) for s, _, o in g.triples((None, PROP['recommendedFor'], None))
    if (s, PROP['contraindicatedFor'], o) in g
]
print(f"Remaining contradictions: {len(remaining)}")


Contradictions found: 75
Resolved: 75
Remaining contradictions: 0


In [20]:
# Save updated KG
g.serialize(destination='C:/kg_project/nutrition_kg_reasoned.ttl', format='turtle')
print("Reasoned KG saved!\n")

# Final summary
foods_count = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Food']))))
nutrients_count = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Nutrient']))))
diseases_count = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Disease']))))
recommended = len(list(g.triples((None, PROP['recommendedFor'], None))))
contraindicated = len(list(g.triples((None, PROP['contraindicatedFor'], None))))

print("=== Final KG Summary ===")
print(f"Foods:                     {foods_count}")
print(f"Nutrients:                 {nutrients_count}")
print(f"Diseases:                  {diseases_count}")
print(f"recommendedFor triples:    {recommended}")
print(f"contraindicatedFor triples:{contraindicated}")
print(f"Total triples:             {len(g)}")

Reasoned KG saved!

=== Final KG Summary ===
Foods:                     365
Nutrients:                 18
Diseases:                  154
recommendedFor triples:    13154
contraindicatedFor triples:307
Total triples:             18394


## 7. KG Embeddings — RotatE
Train RotatE via PyKEEN on the enriched KG to learn vector representations  
of foods, nutrients and diseases. Used for link prediction —  
predicting missing food-disease relationships not captured by logical rules.

In [21]:
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
import torch

# Extract all triples as (subject, predicate, object) strings
triples = []
for s, p, o in g:
    triples.append((
        str(s).split('/')[-1],
        str(p).split('/')[-1],
        str(o).split('/')[-1]
    ))

import numpy as np
triples_array = np.array(triples)
print(f"Total triples for training: {len(triples_array)}")
print("\nSample:")
print(triples_array[:5])

Total triples for training: 18394

Sample:
[['Khorasan_grain_dry_raw' 'recommendedFor' 'Hypertension']
 ['Beans_Dry_Great_Northern_0_moisture' 'recommendedFor'
  'Neural_Tube_Defects']
 ['Flaxseed_ground' 'recommendedFor' 'Muscle_Weakness']
 ['Potatoes_red_without_skin_raw' 'recommendedFor' 'Hypokalemia']
 ['Beans_Dry_Tan_0_moisture' 'recommendedFor' 'Teratozoospermia']]


In [22]:
# Filter out rdf:type triples and keep only semantic ones
meaningful_predicates = ['highIn', 'mediumIn', 'lowIn', 'treats', 
                         'recommendedFor', 'contraindicatedFor', 'worsensWith']

triples_filtered = [t for t in triples if t[1] in meaningful_predicates]
triples_array = np.array(triples_filtered)

print(f"Triples after filtering: {len(triples_array)}")
print("\nSample:")
print(triples_array[:5])
print("\nPredicate distribution:")
predicates = [t[1] for t in triples_filtered]
from collections import Counter
for pred, count in Counter(predicates).most_common():
    print(f"  {pred}: {count}")

Triples after filtering: 17857

Sample:
[['Khorasan_grain_dry_raw' 'recommendedFor' 'Hypertension']
 ['Beans_Dry_Great_Northern_0_moisture' 'recommendedFor'
  'Neural_Tube_Defects']
 ['Flaxseed_ground' 'recommendedFor' 'Muscle_Weakness']
 ['Potatoes_red_without_skin_raw' 'recommendedFor' 'Hypokalemia']
 ['Beans_Dry_Tan_0_moisture' 'recommendedFor' 'Teratozoospermia']]

Predicate distribution:
  recommendedFor: 13154
  mediumIn: 2070
  lowIn: 1064
  highIn: 1055
  contraindicatedFor: 307
  treats: 200
  worsensWith: 7


In [23]:
# Create TriplesFactory
tf = TriplesFactory.from_labeled_triples(triples_array)

# Split into train/test
train, test = tf.split([0.8, 0.2], random_state=42)

print(f"Training triples: {train.num_triples}")
print(f"Test triples:     {test.num_triples}")

Training triples: 14285
Test triples:     3572


In [24]:
result = pipeline(
    model='RotatE',
    training=train,
    testing=test,
    model_kwargs=dict(embedding_dim=64),
    training_kwargs=dict(num_epochs=100, batch_size=256),
    random_seed=42,
)

print("Training done!")
print(result.metric_results.to_df())

No cuda devices were available. The model runs on CPU
C:\kg_project\kg_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training epochs on cpu:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/56.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/56.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/56.0 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/56.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/56.0 [00:00<?, ?batch/s]

Evaluating on cpu:   0%|          | 0.00/3.57k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 1.34s seconds


Training done!
     Side    Rank_type              Metric        Value
0    head   optimistic            variance  1103.985191
1    tail   optimistic            variance    11.195061
2    both   optimistic            variance   583.679571
3    head    realistic            variance  1103.985107
4    tail    realistic            variance    11.195064
..    ...          ...                 ...          ...
220  tail    realistic  adjusted_hits_at_k     0.992275
221  both    realistic  adjusted_hits_at_k     0.912516
222  head  pessimistic  adjusted_hits_at_k     0.832607
223  tail  pessimistic  adjusted_hits_at_k     0.992275
224  both  pessimistic  adjusted_hits_at_k     0.912516

[225 rows x 4 columns]


In [25]:
# Save the trained model
import torch
model = result.model
torch.save(model.state_dict(), 'C:/kg_project/rotate_model.pt')
print("Model saved!")

Model saved!


In [26]:
# Get the most important metrics
metrics = result.metric_results.to_df()

# Filter to the most relevant ones
key_metrics = metrics[
    (metrics['Side'] == 'both') & 
    (metrics['Rank_type'] == 'realistic') &
    (metrics['Metric'].isin(['hits_at_k', 'inverse_harmonic_mean_rank', 'adjusted_hits_at_k']))
]
print(key_metrics.to_string())

     Side  Rank_type                      Metric     Value
95   both  realistic  inverse_harmonic_mean_rank  0.829849
221  both  realistic          adjusted_hits_at_k  0.912516


In [27]:
model = result.model
from pykeen.predict import predict_target

food_to_check = 'Lentils_dry'

predictions = predict_target(
    model=model,
    head=food_to_check,
    relation='recommendedFor',
    triples_factory=tf,
).df

print(f"=== Top disease predictions for {food_to_check} ===")
print(predictions.head(10)[['tail_label', 'score']])

=== Top disease predictions for Lentils_dry ===
                        tail_label     score
267         Hypomagnesemia_primary -1.856736
23             Atrial_Fibrillation -1.856906
516        Vasospasm__Intracranial -2.018551
495  Tachycardia__Supraventricular -2.046448
270                   Infant_Death -2.069545
488  Substance_Withdrawal_Syndrome -2.086390
496       Tachycardia__Ventricular -2.100302
302           Magnesium_Deficiency -2.168909
257                  Hyperglycemia -2.178231
251                     Hemorrhage -2.187452


In [28]:
# Build lookup from clean URI back to original food name
uri_to_original = {}
for _, row in foundation_foods.iterrows():
    clean = clean_uri(row['description'])
    uri_to_original[clean] = row['description']

print(f"Lookup table built: {len(uri_to_original)} foods")

Lookup table built: 365 foods


## 8. Results: Nutrition Knowledge Graph - Dietary Advisor

This tool combines logical reasoning and knowledge graph embeddings 
to provide dietary recommendations based on diseases.

### How to use
Call `full_dietary_advice('Disease_Name')` to get:
- Foods recommended by logical rules (high in nutrients that treat the disease)
- Foods to avoid (high in nutrients that worsen the disease)
- Additional foods predicted by RotatE embeddings

### Available diseases (sample)
- `Anemia`
- `Diabetes_Mellitus`
- `Asthma`
- `Atherosclerosis`
- `Hypertension`
- `Osteoporosis`
- `Obesity`

### Example
`full_dietary_advice('Anemia')`

In [29]:
# List all available diseases correctly
all_diseases = sorted([
    str(o).split('/')[-1].replace('_', ' ')
    for s, p, o in g.triples((None, PROP['treats'], None))
] + [
    str(o).split('/')[-1].replace('_', ' ')
    for s, p, o in g.triples((None, PROP['worsensWith'], None))
])

all_diseases = sorted(set(all_diseases))
print(f"Total diseases in KG: {len(all_diseases)}")
print("\nAll available diseases:")
for d in all_diseases:
    print(f"  {d}")

Total diseases in KG: 154

All available diseases:
  Acrodermatitis
  Acute Kidney Injury
  Alzheimer Disease
  Anemia
  Anemia  Iron Deficiency
  Arrhythmias  Cardiac
  Arthralgia
  Asthma
  Atherosclerosis
  Atrial Fibrillation
  Autistic Disorder
  Basal Cell Nevus Syndrome
  Bone Diseases
  Bone Diseases  Endocrine
  Bradycardia
  Brain Injuries
  COVID 19
  Carcinoma  Ehrlich Tumor
  Carcinoma  Squamous Cell
  Cardiomyopathies
  Cardiotoxicity
  Cardiovascular Diseases
  Cell Transformation  Neoplastic
  Cerebrovascular Disorders
  Chemical and Drug Induced Liver Injury
  Cholestasis
  Cholestasis  Extrahepatic
  Cognition Disorders
  Colitis
  Colonic Neoplasms
  Coronary Vasospasm
  Craniofacial Abnormalities
  Developmental Disabilities
  Diabetes Mellitus
  Diabetes Mellitus  Type 2
  Diabetic Cardiomyopathies
  Diabetic Ketoacidosis
  Diarrhea
  Disease Models  Animal
  Disease Progression
  Disease Susceptibility
  Drug Overdose
  Drug Related Side Effects and Adverse Reacti

In [30]:
# Daily reference values for normalization (based on FDA daily values)
daily_reference = {
    'Sodium':        2300,
    'Potassium':     4700,
    'Calcium':       1300,
    'Iron':            18,
    'Magnesium':      420,
    'Zinc':            11,
    'Vitamin_D':       20,
    'Fiber':           28,
    'Protein':         50,
    'Fat':             78,
    'Carbohydrate':   275,
    'Sugar':           50,
    'Saturated_Fat':   20,
    'Vitamin_C':       90,
    'Vitamin_B6':       1.7,
    'Vitamin_B12':      2.4,
    'Folate':         400,
    'Cholesterol':    300,
}

def score_food_for_disease(food_name, disease_name):
    disease_uri = DISEASE[disease_name.replace(' ', '_')]
    score = 0
    
    food_row = df[df['food'] == food_name]
    
    for _, row in food_row.iterrows():
        nutrient = row['nutrient']
        amount = row['amount']
        nutrient_uri = NUTRIENT[nutrient]
        ref = daily_reference.get(nutrient, 100)
        normalized = amount / ref
        
        if (nutrient_uri, PROP['treats'], disease_uri) in g:
            score += normalized
        if (nutrient_uri, PROP['worsensWith'], disease_uri) in g:
            score -= normalized
    
    return round(score, 4)

In [31]:
def dietary_advice(disease_name):
    """
    Get dietary recommendations for a given disease.
    Use underscores for spaces e.g. 'Diabetes_Mellitus', 'Anemia', 'Hypertension'
    """
    disease_uri = DISEASE[disease_name.replace(' ', '_')]
    
    # From KG reasoning
    # From KG reasoning — now ranked by score
    recommended_raw = [
        uri_to_original.get(s.split('/')[-1], s.split('/')[-1].replace('_', ' '))
        for s, _, _ in g.triples((None, PROP['recommendedFor'], disease_uri))
    ]
    recommended = sorted(
        recommended_raw,
        key=lambda f: score_food_for_disease(f, disease_name),
        reverse=True
    )
    contraindicated_raw = [
        uri_to_original.get(s.split('/')[-1], s.split('/')[-1].replace('_', ' '))
        for s, _, _ in g.triples((None, PROP['contraindicatedFor'], disease_uri))
    ]
    contraindicated = sorted(
        contraindicated_raw,
        key=lambda f: score_food_for_disease(f, disease_name),
        reverse=False  # worst first
    )
    
    # From RotatE predictions
    try:
        predictions = predict_target(
            model=model,
            head=None,
            relation='recommendedFor',
            tail=disease_name.replace(' ', '_'),
            triples_factory=tf,
        ).df
        known = set(s.split('/')[-1] for s, _, _ in g.triples((None, PROP['recommendedFor'], disease_uri)))
        new_preds = predictions[~predictions['head_label'].isin(known)].head(5)
    except:
        new_preds = None

    print(f"╔══════════════════════════════════════════")
    print(f"║ Dietary advice for: {disease_name.replace('_', ' ')}")
    print(f"╠══════════════════════════════════════════")
    print(f"║ ✅ Recommended ({len(recommended)} foods, top 5):")
    for f in recommended[:5]:
        print(f"║    + {f}")
    print(f"║")
    print(f"║ ❌ Avoid ({len(contraindicated)} foods, top 5):")
    for f in contraindicated[:5]:
        print(f"║    - {f}")
    print(f"║")
    print(f"║ 🔮 Additionally predicted by RotatE:")
    if new_preds is not None and len(new_preds) > 0:
        for _, row in new_preds.iterrows():
            print(f"║    ? {row['head_label'].replace('_', ' ')} (score: {row['score']:.3f})")
    else:
        print(f"║    No additional predictions")
    print(f"╚══════════════════════════════════════════")

# Try it out
dietary_advice('Fever')
print()
dietary_advice('Magnesium Deficiency')
print()
dietary_advice('Hypertension')

╔══════════════════════════════════════════
║ Dietary advice for: Fever
╠══════════════════════════════════════════
║ ✅ Recommended (89 foods, top 5):
║    + Egg, yolk, dried
║    + Seeds, pumpkin seeds (pepitas), raw
║    + Sesame butter, creamy
║    + Seeds, sunflower seed kernels, dry roasted, with salt added
║    + Wild rice, dry, raw
║
║ ❌ Avoid (0 foods, top 5):
║
║ 🔮 Additionally predicted by RotatE:
║    ? Nuts pistachio nuts raw (score: -3.347)
║    ? Flour rye (score: -3.445)
║    ? Flour potato (score: -3.529)
║    ? Flour buckwheat (score: -3.532)
║    ? Flour barley (score: -3.551)
╚══════════════════════════════════════════

╔══════════════════════════════════════════
║ Dietary advice for: Magnesium Deficiency
╠══════════════════════════════════════════
║ ✅ Recommended (89 foods, top 5):
║    + Seeds, pumpkin seeds (pepitas), raw
║    + Flaxseed, ground
║    + Seeds, sunflower seed kernels, dry roasted, with salt added
║    + Sesame butter, creamy
║    + Nuts, brazilnuts,

In [32]:
# Show top 10 recommended foods for Hypertension with their scores
disease_name = 'Hypertension'
disease_uri = DISEASE[disease_name]

scored_foods = []
for s, _, _ in g.triples((None, PROP['recommendedFor'], disease_uri)):
    food_name = uri_to_original.get(s.split('/')[-1], s.split('/')[-1].replace('_', ' '))
    score = score_food_for_disease(food_name, disease_name)
    scored_foods.append((food_name, score))

scored_foods.sort(key=lambda x: x[1], reverse=True)

print("Top 10 recommended for Hypertension:")
for food, score in scored_foods[:10]:
    print(f"  {score:.4f}  {food}")

print("\nBottom 5 (worst recommended):")
for food, score in scored_foods[-5:]:
    print(f"  {score:.4f}  {food}")

Top 10 recommended for Hypertension:
  1.5320  Flour, soy, defatted
  1.3715  Chia seeds, dry, raw
  1.3656  Seeds, pumpkin seeds (pepitas), raw
  1.2156  Flaxseed, ground
  1.1981  Flour, soy, full-fat
  1.0916  Nuts, brazilnuts, raw
  1.0413  Flour, coconut
  0.9987  Almond butter, creamy
  0.9977  Sesame butter, creamy
  0.9643  Nuts, almonds, whole, raw

Bottom 5 (worst recommended):
  0.1216  Mushroom, pioppini
  0.1175  Cream, sour, full fat
  0.1171  Mushroom, enoki
  0.1065  Mushroom, crimini
  0.1025  Halibut, frozen, wild caught


In [33]:
# Save final clean KG
g.serialize(destination='C:/kg_project/nutrition_kg_final.ttl', format='turtle')

# Save triples as CSV
triples_clean = []
for s, p, o in g:
    triples_clean.append((
        str(s).split('/')[-1],
        str(p).split('/')[-1],
        str(o).split('/')[-1]
    ))

triples_df = pd.DataFrame(triples_clean, columns=['subject', 'predicate', 'object'])
triples_df.to_csv('C:/kg_project/triples.csv', index=False)

summary = {
    'foods': 365,
    'nutrients': 18,
    'diseases': 154,
    'total_triples': len(g),
    'mrr': 0.83,
    'hits_at_k': 0.91
}

import json
with open('C:/kg_project/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("All files saved cleanly!")
print(f"Final KG: {len(g)} triples")

All files saved cleanly!
Final KG: 18394 triples
